# ATTR v3 — Atom EHR Profiling (Phase 1)

**Scope of this notebook — read before editing anything:**

- This is Phase 1 only: **atom-level EHR data discovery/profiling**, not clinical interpretation.
- For every atom identifier (each keyword, phrase, synonym, ICD code, SNOMED code — checked individually, never pre-merged): does it occur in the EHR, in which table, how often, for how many distinct patients, and what did the real matching text actually look like.
- **Nothing about phenotypes, tiers, buckets, composites, combination rules, temporal rules, or guardrails belongs in this notebook.** That architecture lives in `specialty_configs_v3/` as configuration only, and is a separate, later step.
- Keywords and codes are **never hand-typed in a cell here**. They are loaded from `specialty_configs_v3/atoms/*.json` at runtime.
- We are going **one atom at a time** — prove the profiling logic end-to-end on a single atom before running it across all atoms.

**Table and column names are configured in one place below (Step 1.1).** Nothing else in this notebook should ever hardcode a table or column name.

## Step 1 — Session & Configuration

### 1.0 Active session

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from snowflake.snowpark.context import get_active_session

# Table QC/validation is reused from the existing Steps 1-3 module.
# This notebook does not reuse its wide-net/evidence-table logic (ATTR_WIDE_NET_CANDIDATES,
# ATTR_EVID_*) because those were built from the OLD hardcoded keyword/code lists -
# Phase 1 profiling needs to see the raw source tables directly, unfiltered by them.
HERE = Path.cwd()
CANDIDATE_DIRS = [
    HERE / "specialty_configs",
    HERE,
    Path("/tmp/specialty_configs"),
    Path("/tmp"),
]
_mod_dir = next((p for p in CANDIDATE_DIRS if (p / "rddt_attr_sql.py").exists()), None)
if _mod_dir is None:
    raise FileNotFoundError(
        "rddt_attr_sql.py not found. Searched:\n  " + "\n  ".join(str(p) for p in CANDIDATE_DIRS)
    )
if str(_mod_dir) not in sys.path:
    sys.path.insert(0, str(_mod_dir))
print("rddt_attr_sql.py loaded from:", _mod_dir)

import rddt_attr_sql as rsql
importlib.reload(rsql)

session = get_active_session()
session

In [ ]:
# Change to match our environment
DATABASE = "RDDT"
SCHEMA = "PUBLIC"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
print("Current database:", session.get_current_database())
print("Current schema:", session.get_current_schema())

### 1.1 TABLE CONFIGURATION — edit here only

Every table and column name used anywhere downstream in this notebook comes from this one dict. If a table or column gets renamed, this is the only cell that needs to change.

Same shape as `SOURCE_CONFIG` in `RDDT_ATTR_Snowflake.ipynb` — reused as-is (same logical names, same physical table/column names) since this notebook profiles the same environment.

- `name` — physical table name (add `DB.SCHEMA.` prefix, or set `namespace`, if they live elsewhere)
- `columns` — logical name → physical column name
- `enabled` — set `False` for a table we are not using
- `required` — this table is needed for Phase 1 profiling

In [ ]:
SOURCE_CONFIG = {
    "namespace": None,
    "tables": {
        "census": {
            "name": "CENSUS",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "birth_date": "BirthDate",
                "gender": "Gender",
                "city": "City",
                "state": "State",
                "family_id": "FamilyId",
            },
            "required_columns": (
                "patient_id", "birth_date", "gender", "city", "state", "family_id",
            ),
        },
        "encounter": {
            "name": "ENCOUNTER_VISIT",
            "enabled": True,
            "required": True,
            "columns": {
                "encounter_id": "EncounterId/VisitId",
                "patient_id": "Member/PatientId",
                "encounter_date": "Encounter/Visit Date",
            },
            "required_columns": ("encounter_id", "patient_id", "encounter_date"),
        },
        "claim": {
            "name": "CLAIM",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "diagnosis_code": "DiagnosisCode",
                "other_diagnosis_9": "OtherDiagnosisCodes9",
                "other_diagnosis_10": "OtherDiagnosisCodes10",
                "procedure_code": "ProcedureCode",
                "procedure_modifier_1": "ProcedureModifier1",
                "procedure_modifier_2": "ProcedureModifier2",
                "procedure_modifier_3": "ProcedureModifier3",
                "diagnosis_type": "DiagnosisType",
                "provider_type": "ProviderType",
                "specialty_code": "SpecialtyCode",
                "specialty_name": "SpecialtyName",
                "drg_code": "DRGCode",
                "clinical_notes": "ClinicalNotes",
                "from_date": "FromDate",
                "to_date": "ToDate",
            },
            "diagnosis_columns": ("diagnosis_code", "other_diagnosis_9", "other_diagnosis_10"),
            "procedure_columns": ("procedure_code",),
            "required_columns": (
                "patient_id", "encounter_id",
                "diagnosis_code", "other_diagnosis_9", "other_diagnosis_10",
                "procedure_code", "procedure_modifier_1", "procedure_modifier_2", "procedure_modifier_3",
                "diagnosis_type", "provider_type", "specialty_code", "specialty_name",
                "drg_code", "clinical_notes", "from_date", "to_date",
            ),
        },
        "lab": {
            "name": "LAB",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "lab_id": "LabId",
                "lab_request_id": "LabRequestId",
                "lab_result_id": "LabResultId",
                "observation_identifier": "ObservationIdentifier",
                "observation_value": "ObservationValue",
                "result_status": "ObservationResultStatus",
                "observation_datetime": "ObservationDateTime",
                "lab_result_note": "LabResultNote",
            },
            "required_columns": (
                "lab_id", "encounter_id", "patient_id",
                "lab_request_id", "lab_result_id",
                "observation_identifier", "observation_value", "result_status",
                "observation_datetime", "lab_result_note",
            ),
        },
        "medical_history": {
            "name": "MEDICAL_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "MedicalHistoryId",
                "source_category": "Source/Category",
                "value": "Value",
                "snomed": "SNOMED",
                "secondary_snomed": "Secondary SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "source_category", "value", "snomed", "secondary_snomed", "event_date",
            ),
        },
        "surgical_history": {
            "name": "SURGICAL_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "SurgicalHistoryId",
                "source_category": "Source/Category",
                "value": "Value",
                "snomed": "SNOMED",
                "secondary_snomed": "Secondary SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "source_category", "value", "snomed", "secondary_snomed", "event_date",
            ),
        },
        "family_history": {
            "name": "FAMILY_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "FamilyHistoryId",
                "condition": "Condition",
                "status": "Status",
                "family_member": "FamilyMember",
                "snomed": "SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "snomed", "condition", "status", "family_member", "event_date",
            ),
        },
        "clinical_note": {
            "name": "CLINICAL_NOTE",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "note_id": "NoteId",
                "note_type": "NoteType",
                "note_text": "Clinical Note Text",
                "event_date": "Date",
            },
            "required_columns": (
                "note_id", "encounter_id", "patient_id",
                "note_type", "note_text", "event_date",
            ),
        },
        "social_history": {
            "name": "SOCIAL_HISTORY",
            "enabled": False,
            "required": False,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "value": "Value",
                "event_date": "Date",
            },
            "required_columns": (),
        },
        "medication": {
            "name": "MEDICATION",
            "enabled": False,
            "required": False,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "medication_name": "Medication Name",
                "status": "Medication Status",
                "event_date": "Date",
            },
            "required_columns": (),
        },
    },
}

# Free-text columns eligible for keyword/phrase matching strategies (exact/LIKE/starts/ends).
# Code columns are kept OUT of this dict on purpose -- text strategies must never run on them.
TEXT_COLUMNS = {
    "lab": ("observation_identifier", "observation_value", "lab_result_note"),
    "medical_history": ("value", "source_category"),
    "surgical_history": ("value", "source_category"),
    "family_history": ("condition",),
    "clinical_note": ("note_text",),
    "claim": ("clinical_notes",),
    "social_history": ("value",),
}

# Structured code columns, kept separate from TEXT_COLUMNS so ICD/CPT/SNOMED matching
# (exact IN (...) comparison) never runs on free text, and text strategies never run
# on a code column.
ICD_CODE_COLUMNS = {
    "claim": ("diagnosis_code", "other_diagnosis_9", "other_diagnosis_10"),
}
PROCEDURE_CODE_COLUMNS = {
    "claim": ("procedure_code",),
}
SNOMED_CODE_COLUMNS = {
    "medical_history": ("snomed", "secondary_snomed"),
    "surgical_history": ("snomed", "secondary_snomed"),
    "family_history": ("snomed",),
}

for logical, source in SOURCE_CONFIG["tables"].items():
    flag = "on " if source.get("enabled", True) else "off"
    print(f"[{flag}] {logical:18s} -> {source['name']}")

### 1.2 Validate the configuration against the real tables

Fails immediately if a required table or column is missing, so a rename shows up here instead of as a silently-zero atom result later.

In [ ]:
validation = rsql.validate_sources(session, SOURCE_CONFIG, raise_on_error=True)
validation